In [ ]:
import sys
import json
import ast
import re
from statistics import mean 
print(sys.executable)


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

# The eval makes many small calls, so it runs on a cheaper, faster model.
eval_model = "claude-haiku-4-5"

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "How far away from Earth is the Moon?"
        }
    ]
)

In [ ]:
message.content[0].text

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# model_id lets a call override the global `model` (e.g. use Haiku for the eval).
# max_tokens is 2000 so longer code answers aren't cut off mid-function.
def chat(messages, system=None, stop_sequences=None, model_id=None, max_tokens=2000):
    params = {
        "model": model_id or model,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "How far from Earth is the moon?")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)

### L10: Prompt Engineering

In [ ]:
# prompt = f"""
#     Please provide a solution to the following task:
#     {task}
# """

### L11: Generating Test Datasets

In [ ]:
def parse_json_response(prompt):
    """Ask for JSON and return it parsed.

    The prefilled "```json" makes Claude start straight into the JSON, and the
    "```" stop sequence cuts it off before the closing fence, so what comes
    back is bare JSON that json.loads can read.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"], model_id=eval_model)
    return json.loads(text)

In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
      "task": "Create a Python function to validate an AWS IAM username",
      "format": "python"
  },
  {
      "task": "Create a Python function Parse an s3://bucket/key/path URI into (bucket, key) components.",
      "format": "python"
  },
  {
      "task": "Create a Python function to convert a CloudWatch Logs timestamp (epoch milliseconds) to an ISO 8601 string and back.",
      "format": "python"
  }
]
```

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation

Please generate 3 objects.
"""
    return parse_json_response(prompt)

In [ ]:
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

### L12: Running the eval

In [ ]:
# Same prefill + stop-sequence trick as parse_json_response, so the output is
# bare code with no prose or markdown fences. Otherwise the syntax validators
# in L14 would fail on perfectly good answers.
FENCE_LANG = {"python": "python", "json": "json", "regex": ""}

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    format = test_case["format"].lower()
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with {format}, no commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```" + FENCE_LANG.get(format, ""))
    return chat(messages, stop_sequences=["```"], model_id=eval_model)

### L13: Model based grading

In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    return parse_json_response(eval_prompt)

### L14: Code based grading

In [ ]:
# Code-based graders are all-or-nothing: 10 if the output parses, 0 if not.
# They only check syntax, not whether the code does what the task asked.
# .strip() drops the leading newline left by the prefill.
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"].lower()
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    # Average the two so code that doesn't parse can't score above 5,
    # while the model grade still reflects overall quality.
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [ ]:
# Finally, calculate an average score across all test cases: 
def run_eval(dataset): 
    results = [] 
    for test_case in dataset: 
        result = run_test_case(test_case) 
        results.append(result) 
    average_score = mean([result["score"] for result in results]) 
    print(f"Average score: {average_score}") 
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

for r in results:
    print(f'{r["score"]:>4}  [{r["test_case"]["format"]}]  {r["test_case"]["task"]}')
    print(f'      {r["reasoning"]}\n')

### L15: Prompt evaluator

In [ ]:
# PromptEvaluator isn't in the course files, so it lives in prompt_evaluator.py.
from prompt_evaluator import PromptEvaluator

evaluator = PromptEvaluator(max_concurrent_tasks=5)

In [ ]:
# Named meal_dataset.json so it doesn't overwrite the AWS dataset.json used in L11-L14.
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="meal_dataset.json",
    num_cases=3,
)

In [ ]:
# A deliberately naive baseline prompt.
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages, model_id=eval_model)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="meal_dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
)